# ISM 6564 · Week 6 · Practice — the mechanics of prompting, contracts and LoRA

**ISM 6564 — Text Analytics · Fall 2026 · Week 6: Prompting, Frontier APIs & Fine-Tuning**

## About this practice assignment

- **Not marked, nothing to submit.** It exists so the mechanics are familiar before Assignment 6,
  and the judgement can be the hard part there.
- **Available after Week 6's class** (Wednesday, September 30, 2026, 3:30 PM ET) and open for the
  rest of the term.
- **How to use it.** Work each step in this notebook. When you have tried a step, compare with
  the solution notebook (`ism6564-week06-practice-solution.ipynb`, on Canvas) — only after trying.
- **Time.** About 60–90 minutes of your time. The notebook itself runs in under a minute on an
  Apple-silicon or NVIDIA GPU once your code is right, a few minutes on a CPU. Step 6's short
  training run needs about 4 GB of memory.
- **Model.** `Qwen/Qwen2.5-0.5B-Instruct` (about 1 GB, downloaded once), at full precision. No API
  key is needed anywhere.

| Step | What it rehearses | Lecture slide | Used in Assignment 6 |
| ---: | --- | --- | --- |
| 1 | The chat format: messages become one string, and tokens | *v2: say exactly what you want* | Task 4 (`build_example`) |
| 2 | A contract and a check that never raises | *A hard guardrail: the contract, in code* | Task 1 |
| 3 | What worked examples cost, per 1,000 tickets | *v4*, *Every call is a bill* | the memo (Task 6) |
| 4 | LoRA parameter arithmetic, checked against `peft` | *LoRA: freeze W, learn two thin tables* | Task 4 |
| 5 | Loss masking with `-100` | *The data is the product* | Task 4 |
| 6 | A tiny fine-tune, measured before and after | *The honest evaluation* | Task 4 |

Every step ends in checks. Steps you have not started print a reminder instead of failing, so the
notebook runs top to bottom from the start and you can work the steps in any order.


## Setup


In [ ]:
import importlib.util
import subprocess
import sys

NEEDED = {"pydantic": "pydantic", "tiktoken": "tiktoken", "peft": "peft", "transformers": "transformers"}
_missing = [pkg for mod, pkg in NEEDED.items() if importlib.util.find_spec(mod) is None]
if _missing:
    subprocess.check_call([sys.executable, "-m", "pip", "install", "-q", *_missing])

import json
import os
import random
import re
import time
from typing import Literal

os.environ.setdefault("PYTORCH_ENABLE_MPS_FALLBACK", "1")     # before torch is imported

import numpy as np
import pandas as pd
import torch
from pydantic import BaseModel, Field, ValidationError
from transformers import AutoModelForCausalLM, AutoTokenizer

SEED = 6564
random.seed(SEED)
np.random.seed(SEED)
torch.manual_seed(SEED)
device = torch.device("cuda" if torch.cuda.is_available()
                      else "mps" if torch.backends.mps.is_available() else "cpu")
print(f"torch {torch.__version__}   device: {device}")


class Unfinished(Exception):
    pass


def todo(what: str):
    raise Unfinished(f"{what} is not done yet")


def attempt(label, fn, *args, **kwargs):
    try:
        return fn(*args, **kwargs)
    except Unfinished as exc:
        print(f"Step not done yet [{label}]: {exc}. Fill in the TODO above, then re-run.")
        return None


BASE_URL = ("https://raw.githubusercontent.com/prof-tcsmith/6564F26-DATA/main/"
            "week-06-prompting-apis-finetuning/data")
train = pd.read_csv(f"{BASE_URL}/support_tickets_train.csv", dtype=str).fillna("")
test = pd.read_csv(f"{BASE_URL}/support_tickets_test.csv", dtype=str).fillna("")
CODES = ["SHIP-DELAY", "RET-RMA", "BILL-DISP", "PROD-DEF", "ACCT-SEC", "PROD-INFO"]

MODEL_ID = "Qwen/Qwen2.5-0.5B-Instruct"
tok = AutoTokenizer.from_pretrained(MODEL_ID)
tok.padding_side = "left"
model = AutoModelForCausalLM.from_pretrained(MODEL_ID).to(dtype=torch.float32).to(device).eval()
print(f"train {train.shape}   test {test.shape}   {MODEL_ID} on {device}")


---

## Step 1 — The chat format

An instruct model never sees your list of messages. It sees **one string**, built by the
tokenizer's chat template, with special markers around each turn. Build the two-message
conversation the lecture used — instructions in the **system** message, the ticket in the
**user** message inside `<ticket>` tags — render it, and count its tokens.


In [ ]:
SYSTEM = "You triage support tickets for Northwind Outfitters. Reply with one JSON object."
TICKET = train.text.iloc[0]

# TODO: the two messages (system, then user with the ticket inside <ticket>...</ticket>)
messages = None   # TODO

# TODO: the literal string the model reads (apply_chat_template, not tokenized, with the
# generation prompt added so the model knows it is its turn)
rendered = None   # TODO

# TODO: how many tokens that string is
n_tokens = None   # TODO


In [ ]:
if rendered is None or n_tokens is None:
    print("Step 1 not done yet -- fill in the TODOs above, then re-run.")
else:
    print(rendered)
    print(f"{n_tokens} tokens")
    assert "<|im_start|>system" in rendered and "<|im_start|>user" in rendered, "both turns should be in the string"
    assert rendered.rstrip().endswith("<|im_start|>assistant"), "add_generation_prompt=True ends on the assistant's turn"
    assert "<ticket>" in rendered
    assert n_tokens == len(tok(rendered)["input_ids"]), "n_tokens: count the tokens of `rendered` with tok(...)"
    print("\nStep 1 checks passed. Notice the markers: that is why you never hand-build this string.")


---

## Step 2 — A contract, and a check that never raises

The lecture's contract has four fields plus a confidence. Here, a smaller one to practise on:
**a routing decision** with the queue and a confidence. Write it as a Pydantic model, then a
`check(raw)` that returns `(object, "")` for a valid reply and `(None, reason)` otherwise —
and never raises, whatever the model wrote.


In [ ]:
def extract_json(raw: str) -> dict | None:
    """Provided: the last {...} object in the text, or None."""
    for candidate in reversed(re.findall(r"\{.*?\}", raw, re.S)):
        try:
            obj = json.loads(candidate)
        except json.JSONDecodeError:
            continue
        if isinstance(obj, dict):
            return obj
    return None


# TODO: route_to is one of the six CODES; confidence is a float between 0 and 1 (required).
class Routing(BaseModel):
    pass   # TODO: the two fields


def check(raw: str) -> tuple[Routing | None, str]:
    # TODO: extract_json, then Routing.model_validate inside try/except ValidationError
    todo("check")


In [ ]:
CASES = {
    "clean":            ('{"route_to": "SHIP-DELAY", "confidence": 0.9}', True),
    "fenced":           ('```json\n{"route_to": "RET-RMA", "confidence": 0.6}\n```', True),
    "chatty":           ('Here is my answer: {"route_to": "BILL-DISP", "confidence": 0.7} Hope that helps!', True),
    "made-up code":     ('{"route_to": "BILLING", "confidence": 0.7}', False),
    "confidence 1.3":   ('{"route_to": "PROD-DEF", "confidence": 1.3}', False),
    "confidence text":  ('{"route_to": "PROD-DEF", "confidence": "high"}', False),
    "no confidence":    ('{"route_to": "ACCT-SEC"}', False),
    "prose only":       ("This looks like a shipping problem.", False),
}


def _step2():
    for name, (raw, ok) in CASES.items():
        obj, err = check(raw)
        print(f"{name:<17}{'valid' if obj else 'INVALID':<9}{err[:70]}")
        assert (obj is not None) == ok, f"{name}: expected {'valid' if ok else 'invalid'}"
    return True


if attempt("Step 2", _step2):
    print("\nStep 2 checks passed. The checker turned eight different replies into one of two outcomes.")


---

## Step 3 — What worked examples cost

Few-shot examples are sent, and billed, with every ticket. Count the input tokens of a prompt
with k = 0, 2, 4 and 6 worked examples (taken from `train`), then price each version per
**1,000 tickets** at a provider charging $1.00 per million input tokens and $5.00 per million
output tokens (Claude Haiku 4.5's list price, checked 2026-09-28), with 30 output tokens each.


In [ ]:
import tiktoken

enc = tiktoken.get_encoding("o200k_base")      # OpenAI's tokenizer: an estimate for other providers


def shot_messages(k: int, ticket: str) -> list[dict]:
    """Provided: SYSTEM, then k worked examples as user/assistant turns, then the ticket."""
    msgs = [{"role": "system", "content": SYSTEM}]
    for row in train.head(k).itertuples():
        msgs += [{"role": "user", "content": f"<ticket>\n{row.text}\n</ticket>"},
                 {"role": "assistant", "content": json.dumps({"route_to": row.category, "confidence": 1.0})}]
    return msgs + [{"role": "user", "content": f"<ticket>\n{ticket}\n</ticket>"}]


def input_tokens(msgs: list[dict]) -> int:
    # TODO: the sum of tiktoken token counts of every message's content
    todo("input_tokens")


def cost_per_1000(tokens_in: float, tokens_out: float, price_in: float = 1.00, price_out: float = 5.00) -> float:
    # TODO: US$ for 1,000 tickets; prices are per MILLION tokens
    todo("cost_per_1000")


In [ ]:
def _step3():
    assert abs(cost_per_1000(1000, 0) - 1.0) < 1e-9, "1,000 tickets x 1,000 input tokens at $1 per million is $1.00"
    assert abs(cost_per_1000(0, 1000) - 5.0) < 1e-9, "1,000 tickets x 1,000 output tokens at $5 per million is $5.00"
    assert input_tokens([{"role": "user", "content": "hello world"}]) == len(enc.encode("hello world")), \
        "input_tokens: add up tiktoken's count of every message's content"
    rows = []
    for k in (0, 2, 4, 6):
        n = input_tokens(shot_messages(k, test.text.iloc[0]))
        rows.append((k, n, cost_per_1000(n, 30)))
    for k, n, c in rows:
        print(f"k={k}  input tokens {n:>5}   ${c:.3f} per 1,000 tickets")
    return rows


_rows = attempt("Step 3", _step3)
if _rows:
    print(f"\nSix examples multiply the input by {_rows[-1][1] / _rows[0][1]:.1f}x -- on every ticket.")


---

## Step 4 — LoRA arithmetic

For one weight table W with `d_in` inputs and `d_out` outputs, LoRA learns B (`d_out` × r) and A
(r × `d_in`), so it adds **r × (d_in + d_out)** learned numbers. Compute the adapter size for the
`q_proj` table in every layer of this model at r = 8, then check your number against `peft`.


In [ ]:
def lora_params(d_in: int, d_out: int, r: int) -> int:
    # TODO
    todo("lora_params")


q = model.model.layers[0].self_attn.q_proj                   # one real table
n_layers = model.config.num_hidden_layers
# TODO: the adapter size for q_proj in every layer at r = 8
hand_count = None   # TODO


In [ ]:
from peft import LoraConfig, get_peft_model

if hand_count is None:
    print("Step 4 not done yet -- fill in the TODOs above, then re-run.")
else:
    print(f"q_proj: {q.in_features} -> {q.out_features}, {n_layers} layers")
    _pm = get_peft_model(model, LoraConfig(r=8, lora_alpha=16, target_modules=["q_proj"], task_type="CAUSAL_LM"))
    peft_count = sum(p.numel() for p in _pm.parameters() if p.requires_grad)
    model = _pm.unload()                                     # remove the adapter again for the next steps
    print(f"by hand {hand_count:,}   peft {peft_count:,}")
    assert hand_count == peft_count, "the two counts should match exactly"
    print(f"Step 4 checks passed: {100 * hand_count / sum(p.numel() for p in model.parameters()):.3f}% of the model.")


---

## Step 5 — Loss masking

A training example is the prompt followed by the answer. The labels are the same tokens, except
that every prompt token is **-100**, which the loss ignores, so the model is graded only on the
answer. Write the function that builds the labels.


In [ ]:
def make_labels(prompt_ids: list[int], answer_ids: list[int]) -> list[int]:
    # TODO: -100 for every prompt token, then the answer tokens unchanged
    todo("make_labels")


In [ ]:
MINIMAL = "Route the Northwind ticket. Reply with JSON: {\"route_to\": one of " + "|".join(CODES) + ", \"confidence\": 0-1}"


def example(row) -> tuple[list[int], list[int]]:
    """Provided: prompt ids + answer ids, and the labels from your make_labels."""
    prompt = tok.apply_chat_template([{"role": "system", "content": MINIMAL},
                                      {"role": "user", "content": f"<ticket>\n{row.text}\n</ticket>"}],
                                     tokenize=False, add_generation_prompt=True)
    p_ids = tok(prompt, add_special_tokens=False)["input_ids"]
    a_ids = tok(json.dumps({"route_to": row.category, "confidence": 1.0}) + tok.eos_token,
                add_special_tokens=False)["input_ids"]
    return p_ids + a_ids, make_labels(p_ids, a_ids)


def _step5():
    assert make_labels([1, 2, 3], [4, 5]) == [-100, -100, -100, 4, 5], "make_labels: -100 for each prompt token, then the answer"
    ids, labels = example(train.iloc[0])
    graded = [t for t in labels if t != -100]
    assert len(ids) == len(labels) and labels[0] == -100 and graded, "mask the prompt, keep the answer"
    print(f"{len(ids)} tokens, {len(graded)} graded ({100 * len(graded) / len(ids):.0f}%)")
    print("graded part:", tok.decode(graded))
    return True


if attempt("Step 5", _step5):
    print("\nStep 5 checks passed.")


---

## Step 6 — A tiny fine-tune, measured

Everything is now in place. Score the untuned model on 20 test tickets with the minimal prompt
(how many replies pass your `Routing` check, and how many have the right queue), train a small
adapter (r = 8 on the attention tables) on 240 training tickets for one pass, and score again on
the same 20 tickets. Nothing to write here: run it once Steps 2 and 5 pass, and read the result.

One thing to notice in the tuned replies: every training label says `"confidence": 1.0`, so the
tuned model always writes 1.0. That is the lecture's point about a confidence the model writes
about itself (*A confidence you have to test*): it is text the model learned to produce, not a
measurement.


In [ ]:
@torch.no_grad()
def route_20(m) -> tuple[int, int]:
    """Valid replies and right queues on the first 20 test tickets, minimal prompt, greedy."""
    sub = test.head(20)
    prompts = [tok.apply_chat_template([{"role": "system", "content": MINIMAL},
                                        {"role": "user", "content": f"<ticket>\n{t}\n</ticket>"}],
                                       tokenize=False, add_generation_prompt=True) for t in sub.text]
    valid = right = 0
    for i in range(0, 20, 10):
        enc_ = tok(prompts[i:i + 10], return_tensors="pt", padding=True).to(device)
        out = m.generate(**enc_, max_new_tokens=30, do_sample=False, temperature=None, top_p=None, top_k=None,
                         pad_token_id=tok.pad_token_id)
        for j, g in enumerate(out):
            obj, _ = check(tok.decode(g[enc_["input_ids"].shape[1]:], skip_special_tokens=True))
            valid += obj is not None
            right += obj is not None and obj.route_to == sub.category.iloc[i + j]
    return valid, right


if attempt("Step 6 needs Steps 2 and 5", lambda: (check("{}"), make_labels([1], [2]))) is not None:
    before = route_20(model)
    pm = get_peft_model(model, LoraConfig(r=8, lora_alpha=16, lora_dropout=0.05, task_type="CAUSAL_LM",
                                          target_modules=["q_proj", "k_proj", "v_proj", "o_proj"]))
    data = [example(r) for r in train.sample(240, random_state=SEED).itertuples()]
    opt = torch.optim.AdamW([p for p in pm.parameters() if p.requires_grad], lr=3e-4)
    pm.gradient_checkpointing_enable(gradient_checkpointing_kwargs={"use_reentrant": False})   # saves memory
    pm.enable_input_require_grads()
    model.config.use_cache = False          # the generation cache is useless while training
    pm.train()
    t0 = time.time()
    for i in range(0, len(data), 8):
        batch = data[i:i + 8]
        L = max(len(a) for a, _ in batch)
        ids = torch.tensor([a + [tok.pad_token_id] * (L - len(a)) for a, _ in batch]).to(device)
        lab = torch.tensor([b + [-100] * (L - len(b)) for _, b in batch]).to(device)
        att = torch.tensor([[1] * len(a) + [0] * (L - len(a)) for a, _ in batch]).to(device)
        loss = pm(input_ids=ids, attention_mask=att, labels=lab).loss
        loss.backward()
        opt.step()
        opt.zero_grad(set_to_none=True)
    pm.gradient_checkpointing_disable()
    model.config.use_cache = True
    pm.eval()
    after = route_20(pm)
    print(f"trained {len(data) // 8} steps in {time.time() - t0:.0f}s on {device}")
    print(f"untuned: {before[0]}/20 valid, {before[1]}/20 right queue")
    print(f"tuned:   {after[0]}/20 valid, {after[1]}/20 right queue")
    model = pm.unload()               # remove the adapter, so re-running this step starts from the original model


---

## What you should be able to do now

- Say what the model actually reads for a list of messages, and why you never build that string
  by hand.
- Write a contract and a check that turns any reply into *valid object* or *reason*, without
  crashing.
- Price a prompt per 1,000 calls, and say what worked examples add to that price.
- Compute an adapter's size by hand from the table shapes and r.
- Explain what `-100` in the labels does, and what goes wrong without it.
- Read a before-and-after on the same held-out tickets.

Assignment 6 applies each of these to the full job: the four-field contract, three prompt
versions, the monthly bill, and a LoRA fine-tune on all 720 training tickets.
